In [2]:
import os
import pandas as pd
import shutil

from pathlib import Path

In [5]:
RAW_DATASETS_PATH="../raw-datasets"
TARGET_IMAGE_DATASET_PATH="../data/dataset/images"
TARGET_LABEL_DATASET_PATH="../data/dataset/labels"
ANGELINA_PREFIX="ANG"
DSBI_PREFIX="DSBI"

IGNORE_DIRS = ['.gitignore', 'pics', 'src', 'README.md', 'train.txt', 'val.txt', 'test.txt', 'figures']

In [65]:
print(os.path.basename("C:/braill2/raw-datasets/DSBI-master/data/Massage/M+1+recto.jpg").rsplit("+",1)[1].split(".")[0] == "recto")

True


In [ ]:
def make_dir_if_not_exists(path):
    if not os.path.exists(path):
        os.makedirs(path)

def get_prefix(dataset_name):
    if dataset_name == "AngelinaDataset-master":
        return ANGELINA_PREFIX
    elif dataset_name == "DSBI-master":
        return DSBI_PREFIX
    else:
        raise ValueError(f"Unknown dataset name: {dataset_name}")

def build_dataset(raw_dataset_path, target_image_path, target_label_path):
    dataset_name = os.path.basename(raw_dataset_path)
    prefix = get_prefix(dataset_name)
    
    make_dir_if_not_exists(target_image_path)
    make_dir_if_not_exists(target_label_path)

    for i in os.listdir(raw_dataset_path):
        if i in IGNORE_DIRS:
            continue
        for j in os.listdir(os.path.join(raw_dataset_path, i)):
            if j in IGNORE_DIRS:
                continue    
            for k in os.listdir(os.path.join(raw_dataset_path, i, j)):
                if dataset_name == "AngelinaDataset-master":
                    if k in IGNORE_DIRS:
                        continue
                    if Path(k).suffix in ['.jpg', '.jpeg', '.png']: 
                        shutil.copy(os.path.join(raw_dataset_path, i, j, k), os.path.join(target_image_path, f"{prefix}_{k}"))
                    if Path(k).suffix in ['.csv']: 
                        shutil.copy(os.path.join(raw_dataset_path, i, j, k), os.path.join(target_label_path, f"{prefix}_{k}"))
                if dataset_name == "DSBI-master":
                    if Path(k).suffix in ['.jpg', '.jpeg', '.png'] and os.path.basename(k).rsplit("+",1)[1].split(".")[0] == "recto": 
                        shutil.copy(os.path.join(raw_dataset_path, i, j, k), os.path.join(target_image_path, f"{prefix}_{k}"))
                    if Path(k).suffix in ['.txt'] and os.path.basename(k).rsplit("+",1)[1].split(".")[0] == "recto": 
                        shutil.copy(os.path.join(raw_dataset_path, i, j, k), os.path.join(target_label_path, f"{prefix}_{k}"))


In [42]:
def parse_angelina(label_path):
    rows=[]
    with open(label_path, 'r') as file:
        for line in file:
            line=line.strip()
            if line:
                x1, y1, x2, y2, c = line.split(';')
                width = (float(x2)) - float(x1)    
                height = float(y2) - float(y1)
                x_c = float(x1) + width / 2
                y_c = float(y1) + height / 2

                rows.append([x_c, y_c, width, height, c])
    
    with open("annotation_name.txt", 'w') as file:        
        for row in rows:
            file.write(' '.join(map(str, row)) + '\n')

def parse_dsbi(label_path):
    with open(label_path, 'r') as file:
        lines = file.readlines()

        cols = [int(x) for x in lines[1].strip().split(" ")]
        rows = [int(x) for x in lines[2].strip().split(" ")]

        print(cols)
        print(rows)

        for i in range(3, len(lines)):
            y2 = cols[int(lines[i].strip().split(" ")[0])*2]
            y1 = cols[int(lines[i].strip().split(" ")[0])*2 - 1]

            x2 = rows[int(lines[i].strip().split(" ")[1])*3]
            x1 = rows[int(lines[i].strip().split(" ")[1])*3 - 1]

            dots = lines[i].strip().split(" ")[2:]
        
            width = x2 - x1
            height = y2 - y1
            x_c = x1 + width / 2
            y_c = y1 + height / 2
            print(x_c, y_c, width, height, dots)

In [17]:
parse_angelina("C:/braill2/data/dataset/labels/ANG_0c62b07dc3ff44e4982cbb573f3b6d95.labeled.csv")

In [44]:
parse_dsbi("C:/braill2/raw-datasets/DSBI-master/data/Fundamentals of Massage/FM+1+recto.txt")

[47, 67, 94, 114, 142, 162, 189, 209, 236, 256, 284, 308, 336, 361, 389, 410, 440, 462, 493, 516, 544, 566, 596, 616, 648, 672, 700, 722, 753, 775, 806, 828, 854, 874, 909, 929, 960, 981, 1013, 1031, 1063, 1085, 1114, 1137, 1165, 1185, 1215, 1235, 1262, 1282, 1309, 1329, 1356, 1376, 1404, 1424, 1451, 1471, 1498, 1518, 1546, 1566, 1593, 1613, 1640, 1660]
[47, 67, 87, 125, 145, 165, 203, 223, 243, 281, 302, 323, 362, 384, 405, 446, 468, 489, 527, 547, 567, 611, 632, 653, 691, 711, 731, 775, 797, 818, 856, 876, 896, 934, 954, 974, 1012, 1032, 1052, 1090, 1110, 1130, 1168, 1188, 1208, 1246, 1266, 1286, 1324, 1344, 1364, 1402, 1422, 1442, 1480, 1500, 1520, 1558, 1578, 1598, 1636, 1656, 1676, 1714, 1734, 1754, 1792, 1812, 1832, 1930, 1951, 1972, 2009, 2029, 2049, 2087, 2107, 2127, 2165, 2185, 2205, 2243, 2263, 2283]
508.0 222.5 38 27 ['1', '0', '1', '0', '0', '0']
589.0 222.5 44 27 ['1', '1', '1', '0', '0', '1']
672.0 222.5 38 27 ['1', '1', '0', '1', '1', '0']
753.0 222.5 44 27 ['1', '0', '1

In [6]:
for i in os.listdir(RAW_DATASETS_PATH):
    if i in IGNORE_DIRS:
        continue
    build_dataset(os.path.join(RAW_DATASETS_PATH, i), TARGET_IMAGE_DATASET_PATH, TARGET_LABEL_DATASET_PATH)